In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
from os import path
import matplotlib.ticker as ticker
from scipy import stats
from sklearn.linear_model import LinearRegression

labels = ["none", "paiceHusk", "krovetz", "sStripping", "porter2", "lovins", "wikt"]
labels_title = ["None", "Paice/Husk", "Krovetz", "S-Stripping", "Porter2", "Lovins", "Wikt"]

home_dir = "/home/harka424/Documents/COSC490/Stemming"
data_dir_clueweb = path.join(home_dir, "Data/clueweb")
table_output = f"{home_dir}/ClueWeb/tables/queryBreakdown_table.tex"

In [2]:
stem_df = pd.read_csv(f"{data_dir_clueweb}/stem_count.csv")
idf_df = pd.read_csv(f"{data_dir_clueweb}/idf.csv")
ndcg_df = pd.read_csv(f"{data_dir_clueweb}/ndcg_all",sep=" ")

In [3]:
display(ndcg_df)

,stemmer,qid,ndcg,n,collectionSize,maxLength
0,paiceHusk,201,0.406542,1,51638042,2000000
1,paiceHusk,202,0.122014,1,51638042,2000000
2,paiceHusk,203,0.532601,1,51638042,2000000
3,paiceHusk,204,0.877115,1,51638042,2000000
4,paiceHusk,205,0.289510,1,51638042,2000000
...,...,...,...,...,...,...
702,none,297,0.745453,1,51638042,2000000
703,none,298,0.876001,1,51638042,2000000
704,none,299,0.444044,1,51638042,2000000
705,none,300,0.000000,1,51638042,2000000


In [4]:
queries = {
    "228": ["hawaiian", "volcano", "observatories"],
    "240": ["presidential", "middle", "names"],
    "293": ["educational", "advantages", "of", "social", "networking", "sites"],
}

In [10]:
with open(table_output, "w") as f:

    f.write("\\begin{table}[H]\n"
            "\centering\n"
            "\caption{Comparison of Stemming Output on the ClueWeb.}\n"
            "\label{tab:queryBreakdown}\n"
            "\\resizebox{\\textwidth}{!}{%\n"
            "\\begin{tabular}{@{}lll@{ }ll@{ }ll@{ }ll@{ }ll@{ }ll@{ }ll@{ }l@{}}\n" \
            "\\toprule\n"
            " & QID & \multicolumn{2}{c}{None} & \multicolumn{2}{c}{Paice/Husk} & \multicolumn{2}{c}{Krovetz}  & \multicolumn{2}{c}{S-Stripping}  & \multicolumn{2}{c}{Porter2}  & \multicolumn{2}{c}{Lovins} & \multicolumn{2}{c}{Wikt} \\\\ \midrule \n"
            )
    
    for qid in queries:
        query = queries[qid]
        for i in range(0,len(query)):
            line = ""
            if i < 1:
                line += (f" & {qid} & ")
            else:
                line += (f" &  & ")

            #none isnt in the stem map
            idf = float(idf_df[(idf_df["stemmer"]=="none") & (idf_df["word"]==query[i])].loc[:,"idf"].values[0])
            line += f"{query[i]} & {idf:.2f} "

            for stemmer in labels[1:]:
                idf = float(idf_df[(idf_df["stemmer"]==stemmer) & (idf_df["word"]==query[i])].loc[:,"idf"].values[0])
                stem = stem_df[(stem_df["stemmer"]==stemmer) & (stem_df["word"]==query[i])].loc[:,"stem"].values[0]
                line += f" & {stem}  & {idf:.2f}"
            line += " \\\\"

            #if this is the last term in the query, add midrule
            if i == len(query)-1:
                line += " \midrule"
            line += "\n"
            f.write(line)
        
        #add the ndcg for the query
        line = "nDCG@10 & "
        for stemmer in labels:
            ndcg = ndcg_df[(ndcg_df["stemmer"]==stemmer)&(ndcg_df["qid"]==qid)].loc[:,"ndcg"].values[0]
            line += " & \multicolumn{2}{c}{" + f"{ndcg:.4f}" + "}"

        #hardcoding this tbh
        if qid == "293":
            line += "\\\\ \\bottomrule \n"
        else:
            line += "\\\\ \midrule \n"
        f.write(line)

    f.write("\end{tabular}%\n"
            "}\n"
            "\end{table}\n")

    

### idf and cluster counts

In [7]:
with open(table_output, "w") as f:

    f.write("\\begin{sidewaystable}\n"
            "\centering\n"
            "\caption{Comparison of Stemming Output on the ClueWeb.}\n"
            "\label{tab:queryBreakdown}\n"
            "\\resizebox{\\textwidth}{!}{%\n"
            "\\begin{tabular}{@{}llc@{ }rc@{ }rrc@{ }rrc@{ }rrc@{ }rrc@{ }rrc@{ }rr@{}}" \
            "\\toprule\n"
            " & QID & \multicolumn{2}{c}{None} & \multicolumn{3}{c}{Paice/Husk} & \multicolumn{3}{c}{Krovetz}  & \multicolumn{3}{c}{S-Stripping}  & \multicolumn{3}{c}{Porter2}  & \multicolumn{3}{c}{Lovins} & \multicolumn{3}{c}{Wikt} \\\\ \n"
            " & & stem & idf & stem & count & idf & stem & count & idf & stem & count & idf & stem & count & idf & stem & count & idf & stem & count & idf \\\\ \midrule \n"
            )
    
    for qid in queries:
        query = queries[qid]
        for i in range(0,len(query)):
            line = ""
            if i < 1:
                line += (f" & {qid} & ")
            else:
                line += (f" &  & ")

            #none doesnt have clusters
            idf = float(idf_df[(idf_df["stemmer"]=="none") & (idf_df["word"]==query[i])].loc[:,"idf"].values[0])
            line += f"{query[i]} & {idf:.2f} "

            for stemmer in labels[1:]:
                count = stem_df[(stem_df["stemmer"]==stemmer) & (stem_df["word"]==query[i])].loc[:,"count"].values[0]
                idf = float(idf_df[(idf_df["stemmer"]==stemmer) & (idf_df["word"]==query[i])].loc[:,"idf"].values[0])
                stem = stem_df[(stem_df["stemmer"]==stemmer) & (stem_df["word"]==query[i])].loc[:,"stem"].values[0]
                line += f" & {stem} & {count} & {idf:.2f}"
            line += " \\\\"

            #if this is the last term in the query, add midrule
            if i == len(query)-1:
                line += " \midrule"
            line += "\n"
            f.write(line)
        
        #add the ndcg for the query
        ndcg = ndcg_df[(ndcg_df["stemmer"]=="none")&(ndcg_df["qid"]==qid)].loc[:,"ndcg"].values[0]
        line = "nDCG@10 & & \multicolumn{2}{c}{" + f"{ndcg:.4f}" + "}"
        for stemmer in labels[1:]:
            ndcg = ndcg_df[(ndcg_df["stemmer"]==stemmer)&(ndcg_df["qid"]==qid)].loc[:,"ndcg"].values[0]
            line += " & \multicolumn{3}{c}{" + f"{ndcg:.4f}" + "}"

        #hardcoding this tbh
        if qid == "293":
            line += "\\\\ \\bottomrule \n"
        else:
            line += "\\\\ \midrule \n"
        f.write(line)

    f.write("\end{tabular}%\n"
            "}\n"
            "\end{sidewaystable}\n")

    